##Ingesting Fcat table into bronze layer

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, IntegerType

from pyspark.sql import functions as F

In [0]:
catalogue_name = 'ecommerce'

In [0]:
order_items_schema = StructType([
    StructField('dt', StringType(), True), 
    StructField('order_ts', StringType(), True), 
    StructField('customer_id', StringType(), True), 
    StructField('order_id', StringType(), True), 
    StructField('item_seq', IntegerType(), True), 
    StructField('product_id', StringType(), True), 
    StructField('quantity', StringType(), True), 
    StructField('unit_price_currency', StringType(), True), 
    StructField('unit_price', StringType(), True), 
    StructField('discount_pct', StringType(), True), 
    StructField('tax_amount', StringType(), True), 
    StructField('channel', StringType(), True), 
    StructField('coupon_code', StringType(), True)
])

raw_data = "/Volumes/ecommerce/source_data/raw/order_items/landing/*.csv"

df_fact = spark.read.option("delimeter", ",").csv(raw_data, schema=order_items_schema,header=True)\
                .withColumn("_source_file",F.col("_metadata.file_path"))\
                .withColumn("_ingested_at", F.current_timestamp())

In [0]:
df_fact.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalogue_name}.bronze.brnz_order_items")